<a id="top"></a>

# Instrument the graph and compare two traces

```yaml
title: "Instrument the graph and compare two traces"
keywords: tracing, instrumentation, spans, TraceEvent, trace diff, signal vs noise, jsonl, non-determinism, teacher lecture
estimated duration: 25
```

> **Lesson:** L12 Tracing. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md) (objectives 3 & 4).
>
> Runs offline — no API key needed (scripted FakeModel).

## Contents

- [1. Setup — build three traces](#1-setup--build-three-traces)
- [2. Instrumentation is a wrapper, not a rewrite](#2-instrumentation-is-a-wrapper-not-a-rewrite)
  - [2.1 The run already carries its trace](#21-the-run-already-carries-its-trace)
  - [2.2 The field set, and what is left out](#22-the-field-set-and-what-is-left-out)
  - [2.3 A trace is data — JSONL round-trip](#23-a-trace-is-data--jsonl-round-trip)
- [3. Compare two traces of the same task](#3-compare-two-traces-of-the-same-task)
  - [3.1 By eye first](#31-by-eye-first)
  - [3.2 A small typed diff helper](#32-a-small-typed-diff-helper)
  - [3.3 Signal vs noise](#33-signal-vs-noise)
- [4. Takeaways](#4-takeaways)
- [5. Common pitfalls: L12's four anti-patterns](#5-common-pitfalls-l12s-four-anti-patterns)

## 1. Setup — build three traces

We trace the **same graph** from L10 — `agent_graph.run(...)` — but here we drive it with a scripted `FakeModel` instead of a live Sonnet 4.6 client. That keeps the run deterministic and keyless, so every student sees the *exact same trace*. The live version instruments against the real client; the only thing that changes is the model object passed in.

We build three runs:

- **`good`** — a clean two-tool run that terminates naturally.
- **`run_a`** — a runaway: it keeps asking for the same failing tool until it hits the `max_steps` cap.
- **`run_b`** — the *same task* as `run_a` after a "tightened prompt", now giving up cleanly (natural termination).

`run_a` vs `run_b` is our before/after-a-prompt-edit A/B pair in Section 3.

In [1]:
# Setup: imports + the three scripted runs. Runs top-to-bottom, no edits, no API key.
from pathlib import Path

from fluffy_potato_curriculum.common.agent_graph import RunResult, run
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS
from fluffy_potato_curriculum.common.tracing import (
    TraceEvent,
    from_jsonl,
    read_jsonl,
    to_jsonl,
    write_jsonl,
)

# --- good: a clean two-tool run that terminates naturally ---
good_model = FakeModel(
    [
        tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
        tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
        text_reply("17 * 23 is 391, and Tokyo has about 37,000,000 people."),
    ]
)
good: RunResult = run(good_model, TOOLS, "What is 17 * 23, and what is the population of Tokyo?")

# --- run_a: a runaway. The model keeps asking lookup('Atlantis'); the distinct call
#     ids (r0, r1, ...) keep the graph's add_messages reducer from de-duping the
#     repeats, so it cycles until the recursion cap (max_steps=4) forces a halt. ---
run_a_model = FakeModel(
    [tool_reply(tool_call(f"r{i}", "lookup", {"city": "Atlantis"})) for i in range(8)]
)
run_a: RunResult = run(run_a_model, TOOLS, "What is the population of Atlantis?", max_steps=4)

# --- run_b: the SAME task after a "tightened prompt". Now the model gives up
#     cleanly with text only -> natural termination. ---
run_b_model = FakeModel([text_reply("I don't have a population on file for Atlantis.")])
run_b: RunResult = run(run_b_model, TOOLS, "What is the population of Atlantis?")

print("good :", good.termination, good.iterations, "iterations")
print("run_a:", run_a.termination, run_a.iterations, "iterations")
print("run_b:", run_b.termination, run_b.iterations, "iterations")

good : natural 3 iterations
run_a: max_steps 4 iterations
run_b: natural 1 iterations


[↑ Back to top](#top)

## 2. Instrumentation is a wrapper, not a rewrite

This is the headline of objective 3. The L10 graph's control flow is **correct and untouched** — and because the graph is *compiled*, you can't reach inside it anyway. It still calls the model, dispatches tools through the `ToolNode`, and stops on natural/`max_steps` exactly as before. Tracing only *observes* it: it consumes the graph's `stream` and turns each `{node: update}` chunk into one `TraceEvent` (a "span") in a list. If adding a trace ever changed *how the graph decides things*, observation would have leaked into control flow — and that's the bug to watch for.

So we did **not** rewrite the graph to trace it. `agent_graph.run(...)` already emits a trace; we just read `.trace` off the result.

### 2.1 The run already carries its trace

`run(...)` returns a `RunResult` whose `.trace` is an ordered `list[TraceEvent]`. Walk the span boundaries:

- one **`chain`** span first (`name="agent_graph.run"`) — the framing span for the whole run, carrying the termination cause up top;
- one **`llm`** span per `agent`-node visit (`name="agent"`, token `usage` set, `outputs={"tool_calls": [...]}`);
- one **`tool`** span per `ToolNode` dispatch (`name=` the tool name, `inputs=` the args the model chose, `outputs={"content": ...}`, `error` set on failure).

All spans of one run share a single `trace_id`. Let's narrate the `good` run from its trace alone.

In [2]:
# One span per boundary, in order. .one_line() is a compact human rendering.
print(f"{len(good.trace)} spans, all sharing trace_id={good.trace[0].trace_id[:8]}...\n")
for span in good.trace:
    print(span.one_line())

6 spans, all sharing trace_id=e50d6f16...

chain agent_graph.run      {'termination': 'natural', 'iterations': 3}
llm   agent                -> tool_calls=['calculator'] tokens=120
tool  calculator           {'expression': '17*23'} -> '391'
llm   agent                -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Tokyo'} -> '37000000'
llm   agent                -> tool_calls=[] tokens=120


Read it top to bottom: the run terminated `natural` in 3 iterations; on its first `agent`-node visit the model called `calculator(expression='17*23')` and got `391`; on the second it called `lookup(city='Tokyo')` and got `37000000`; on the third it returned text and stopped. **Notice we reconstructed the whole run without re-executing it** — the trace *is* the reproduction.

You can also point at where each `RunResult` summary field came from: `termination` is on the `chain` span's `outputs`, `iterations` is the count of `llm` spans, and `final_text` is the last `llm` span's text.

### 2.2 The field set, and what is left out

The `TraceEvent` field set is deliberately small and defensible — **each field answers a question you actually ask of a run**:

- `start_time` / `end_time` (→ `.duration_s`) — *why was it slow?* (latency)
- `inputs` (the tool args) — *why did it loop?* (repeated args make a runaway obvious)
- `usage` (token counts on `llm` spans) — *why did it cost that much?*
- `run_type` / `name` — *what kind of step, and which one?*
- `outputs` / `error` — *what came back, and did it fail?*

Just as important is what's **deliberately left out**: full prompt bodies, raw tracebacks, and every intermediate variable. A trace bloated with everything is as unreadable as no trace — minimalism is a design choice, not an omission. Here's one `llm` span and one `tool` span in full.

In [3]:
# The first llm span and the first tool span, dumped in full.
first_llm = next(s for s in good.trace if s.run_type == "llm")
first_tool = next(s for s in good.trace if s.run_type == "tool")

print("an llm span:")
print(first_llm.model_dump())
print("\na tool span:")
print(first_tool.model_dump())

an llm span:
{'run_id': 'ca7e97445e9346aabbd9880989325c84', 'trace_id': 'e50d6f16da71442f833d626edfa13204', 'parent_run_id': None, 'run_type': 'llm', 'name': 'agent', 'inputs': {}, 'outputs': {'tool_calls': ['calculator']}, 'error': None, 'start_time': 1783196668.947974, 'end_time': 1783196668.9489121, 'usage': {'input_tokens': 100, 'output_tokens': 20}}

a tool span:
{'run_id': '0447ba05ae9a4abfbac401b6ff2d86ac', 'trace_id': 'e50d6f16da71442f833d626edfa13204', 'parent_run_id': None, 'run_type': 'tool', 'name': 'calculator', 'inputs': {'expression': '17*23'}, 'outputs': {'content': '391'}, 'error': None, 'start_time': 1783196668.9489121, 'end_time': 1783196668.9494379, 'usage': None}


### 2.3 A trace is data — JSONL round-trip

Because a trace is just a list of typed records, it serializes cleanly to **JSON-lines** (one span per line) and parses straight back. This is what makes a trace *durable* — you can write it to disk now and read it an hour or a hundred runs later, no re-execution. `to_jsonl` / `from_jsonl` go through text; `write_jsonl` / `read_jsonl` go through a file.

In [4]:
# One span per line. Show the first two lines, then prove a full round-trip.
jsonl = to_jsonl(good.trace)
print("first two JSONL lines:")
for line in jsonl.splitlines()[:2]:
    print(" ", line)

# text round-trip: serialize -> parse -> identical spans.
reparsed = from_jsonl(jsonl)
assert reparsed == good.trace

# file round-trip via a temp path (write_jsonl / read_jsonl).
tmp = Path("good_trace.jsonl")
write_jsonl(good.trace, tmp)
assert read_jsonl(tmp) == good.trace
tmp.unlink()  # clean up the demo artifact

print(f"\nround-trip OK: {len(reparsed)} spans recovered, identical to the original")

first two JSONL lines:
  {"run_id":"e50d6f16da71442f833d626edfa13204","trace_id":"e50d6f16da71442f833d626edfa13204","parent_run_id":null,"run_type":"chain","name":"agent_graph.run","inputs":{"user_msg":"What is 17 * 23, and what is the population of Tokyo?"},"outputs":{"termination":"natural","iterations":3},"error":null,"start_time":1783196668.947974,"end_time":1783196668.950135,"usage":null}
  {"run_id":"ca7e97445e9346aabbd9880989325c84","trace_id":"e50d6f16da71442f833d626edfa13204","parent_run_id":null,"run_type":"llm","name":"agent","inputs":{},"outputs":{"tool_calls":["calculator"]},"error":null,"start_time":1783196668.947974,"end_time":1783196668.9489121,"usage":{"input_tokens":100,"output_tokens":20}}

round-trip OK: 6 spans recovered, identical to the original


[↑ Back to top](#top)

## 3. Compare two traces of the same task

Objective 4: run the *same task* twice and diff the traces to see what changed. Our pair is `run_a` (the runaway) and `run_b` (the same Atlantis task after a "tightened prompt"). The before/after question is concrete: **did tightening the prompt actually stop the runaway loop, or did I just get a lucky run?**

### 3.1 By eye first

Cheapest first move: print both traces with `.one_line()` and read them side by side. Do the eyeball diff before reaching for any helper.

In [5]:
def print_trace(label: str, result: RunResult) -> None:
    """Print a labeled trace, one span per line."""
    print(f"=== {label}: {result.termination} in {result.iterations} iterations ===")
    for span in result.trace:
        print(span.one_line())


print_trace("run_a (before)", run_a)
print()
print_trace("run_b (after tightened prompt)", run_b)

=== run_a (before): max_steps in 4 iterations ===
chain agent_graph.run      {'termination': 'max_steps', 'iterations': 4}
llm   agent                -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
llm   agent                -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
llm   agent                -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
llm   agent                -> tool_calls=['lookup'] tokens=120
tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]

=== run_b (after tig

By eye: `run_a` calls `lookup('Atlantis')` four times — each a `[is_error]` — and ends in `max_steps`. `run_b` makes **zero** tool calls and ends `natural`. The runaway is gone. The eyeball diff works fine for two short traces; it stops scaling the moment traces get long or you have dozens of runs — which is exactly why we turn the comparison into *data*.

### 3.2 A small typed diff helper

The eyeball version upgrades to a ~10-line helper that compares three things off the trace data:

1. the **tool-call trajectory** — `[(s.name, s.inputs) for s in trace if s.run_type == "tool"]`;
2. the **termination** cause — read off the `chain` span's `outputs`;
3. the **total tokens** — sum of `usage.total_tokens` over the `llm` spans.

It reports each as changed or unchanged. This is "trace is data" made concrete: the diff is just list/dict comparison.

In [6]:
def trajectory(trace: list[TraceEvent]) -> list[tuple[str, dict[str, object]]]:
    """The ordered (tool_name, args) pairs the model chose."""
    return [(s.name, s.inputs) for s in trace if s.run_type == "tool"]


def termination_of(trace: list[TraceEvent]) -> str:
    """The termination cause, read off the chain span's outputs."""
    chain = next(s for s in trace if s.run_type == "chain")
    return str(chain.outputs["termination"])


def total_tokens(trace: list[TraceEvent]) -> int:
    """Total tokens across all llm spans."""
    return sum(s.usage.total_tokens for s in trace if s.usage is not None)


def diff_traces(a: list[TraceEvent], b: list[TraceEvent]) -> dict[str, object]:
    """Compare two traces of the same task on trajectory, termination, and tokens.

    Example output:
        {"trajectory_changed": True,
         "termination": ("max_steps", "natural"),
         "tokens": (480, 120)}
    """
    return {
        "trajectory_changed": trajectory(a) != trajectory(b),
        "termination": (termination_of(a), termination_of(b)),
        "tokens": (total_tokens(a), total_tokens(b)),
    }


diff = diff_traces(run_a.trace, run_b.trace)
print(diff)

{'trajectory_changed': True, 'termination': ('max_steps', 'natural'), 'tokens': (480, 120)}


### 3.3 Signal vs noise

The diff flags three changes. Which are **signal** (a real behavior change) and which would be **noise** (run-to-run variation that means nothing)?

- **Trajectory + termination: signal.** Four failing `lookup('Atlantis')` calls ending in `max_steps`, versus zero tool calls ending `natural`, is a real behavioral fix — the runaway is gone. That's the regression-or-fix you care about.
- **Token total: usually noise.** Tokens (and latency) jitter run to run even on identical behavior; here they moved only *because* the trajectory changed. On a non-deterministic model, a small token/latency delta on its own proves nothing.

And the crucial caveat: **a single differing run can't prove a fix.** The graph is non-deterministic — `run_a` might have been an unlucky path and `run_b` a lucky one. One comparison is suggestive, not conclusive; you'd need the same task across *several* runs to trust it. That need — many cases, judged repeatably — is the **seed of L13's eval set.**

[↑ Back to top](#top)

## 4. Takeaways

- **Instrumentation is a wrapper, not a rewrite.** The L10 graph's control flow is untouched — you can't reach inside a compiled graph anyway; tracing only *observes* it by turning each streamed `{node: update}` chunk into one span (`chain` / `llm` / `tool`). If a trace ever changed the run's behavior, observation leaked into control flow.
- **The field set is small and defensible.** Each field answers a question — latency → why slow, repeated args → why looping, tokens → why expensive — and full prompt bodies, tracebacks, and every variable are left out *on purpose*.
- **A trace is data.** It round-trips through JSONL unchanged, so it's durable and diffable — you read a past run instead of re-running a non-deterministic agent.
- **Diff = signal vs noise.** A real trajectory/termination change is signal; token/latency jitter is noise. **One differing run proves nothing** — that's the seed of L13's eval set.
- **Next:** [`L1205_lab`](L1205_lab_empty.ipynb) has you write `tool_trajectory` and `diff_traces` yourself; then [`L1206_lecture`](L1206_lecture.md) sends the *same* spans to a real Langfuse dashboard. Frame L12 as **"produce the record"** and L13 as **"judge the record."**

[↑ Back to top](#top)

## 5. Common pitfalls: L12's four anti-patterns

The failure modes above were shown across the demos — name them now so you can catch yourself when
you instrument your *own* agent. All four are one instrument-deliberately discipline: a trace is a
**curated record**, neither a firehose nor a shrug.

| Anti-pattern | Cure | Where you saw it |
| --- | --- | --- |
| **Tracing too little** — a wall of `print()` with no structure, order, or `trace_id`; you can't filter, diff, or locate a failure. | Structure + order + one shared run id — the minimum that makes a run *readable after the fact*. | [`L1202`](L1202_lecture.ipynb) (`print()` vs. a structured record); §1 here builds the structured version |
| **Tracing too much** — full prompt bodies, raw tracebacks, every variable, and PII dumped into spans until the trace is a cost/privacy liability. | The minimal field set where each field answers a question; instrumentation is deliberate *subtraction*. | §2.2 "the field set, and what is left out" |
| **Not tracing tool arguments** — recording that `lookup` was *called* but not with *what*, so the wrong-arguments bug is invisible. | Trace `inputs`, not just call names — a success flag never reveals a call to the *wrong question*. | §3.1 — the repeated `lookup('Atlantis')` args are what expose the runaway |
| **Reading the summary, not the trace** — trusting a success flag or a `natural` stop as "correct", or re-running a non-deterministic agent to "reproduce" a bug. | Read the arguments and the model's *next* span; the captured trace **is** the reproduction — don't chase it live. | §2 (reconstructed without re-executing) + §3.3 |

- **#1 and #2 are the same dial set wrong in both directions** — getting the field set right (§2.2)
  *is* the instrumentation skill.
- **#3 and #4 are the reading half — and they are L13's first eval cases:** each failure you can
  *read* here becomes a regression case you can *score* next lesson. That is the L12 → L13 handoff.

[↑ Back to top](#top)